# A Simple RAG Demo Project

Before running the notebook, follow the setup instructions in [README.md](README.md).

In [1]:

# Run once when setting up the project in Colab
#!git clone https://github.com/raivisskadins/simple-rag-demo.git
#%cd simple-rag-demo
#!git pull

# Run once if packages are missing (required also for Colab)
#!pip install -r requirements.txt

# Run to pull latest changes from GitHub
#!git pull


In [2]:
# ==============================
# 1. Imports
# ==============================

import os
import sys
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv(override=True)

client = OpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT")
)

chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

In [3]:
# ==============================
# 2. Load environment variables
# ==============================

# Load environment variables
import os
import sys
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    os.environ["AZURE_OPENAI_API_KEY"] = userdata.get("AZURE_OPENAI_API_KEY")
    os.environ["AZURE_OPENAI_ENDPOINT"] = userdata.get("AZURE_OPENAI_ENDPOINT")
    os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"] = userdata.get("AZURE_OPENAI_CHAT_DEPLOYMENT")
else:
    load_dotenv(override=True)

client = OpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT")
)

chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

os.environ["HF_HOME"] = r"E:\huggingface_cache"
os.environ["TRANSFORMERS_CACHE"] = r"E:\huggingface_cache"
os.environ["SENTENCE_TRANSFORMERS_HOME"] = r"E:\huggingface_cache"

embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

print("Endpoint:", os.getenv("AZURE_OPENAI_ENDPOINT"))
print("Deployment:", chat_deployment)
print("Key exists:", os.getenv("AZURE_OPENAI_API_KEY") is not None)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Endpoint: https://ag23098-7581-resource.services.ai.azure.com/openai/v1/
Deployment: gpt-4o
Key exists: True


In [4]:
# ==============================
# 3. Load knowledge base
# ==============================

with open("./data/knowledge.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Each paragraph becomes one chunk
chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

print("Chunks:")
for c in chunks:
    print("-", c)


Chunks:
- Humans only use 2% of their brains, the Great Wall of China is visible from the Moon, drinking eight liters of water a day is essential for survival, and lightning never strikes the same place twice.
- Mount Everest is the tallest mountain on Earth. Its peak is 8848 meters above sea level and it is part of the Himalayas.
- The Amazon rainforest is the largest tropical rainforest in the world. It spans multiple countries in South America and contains immense biodiversity.
- The Pacific Ocean is the largest ocean on Earth. It covers more than 30% of the planet’s surface.


In [5]:
# ==============================
# 4. Create embeddings
# ==============================

def get_embedding(text):
    emb = embedding_model.encode(text)
    emb = emb / np.linalg.norm(emb)
    return emb

chunk_embeddings = np.array(
    [get_embedding(chunk) for chunk in chunks]
).astype("float32")
embedding_dim = chunk_embeddings.shape[1]



# ==============================
# 5. Create FAISS index
# ==============================

index = faiss.IndexFlatIP(embedding_dim)
# remove previously added content
index.reset()
index.add(chunk_embeddings)

print("FAISS index size:", index.ntotal)


FAISS index size: 4


In [ ]:
# ==============================
# 6–10. Test several questions
# ==============================

test_questions = [
    "Is the Great Wall of China visible from the Moon?",
    "What is the tallest mountain on Earth?",
    "Which rainforest is the largest tropical rainforest in the world?",
    "What is the largest ocean on Earth?",
    "What is the capital city of Latvia?"
]

k = 2

for question in test_questions:
    print("=" * 80)
    print("Question:", question)

    # Convert question to embedding
    question_embedding = np.array(
        [get_embedding(question)]
    ).astype("float32")

    # Search for relevant chunks in FAISS
    distances, indices = index.search(question_embedding, k)

    # Get retrieved chunks
    retrieved_chunks = [chunks[i] for i in indices[0]]

    print("\nRetrieved context:")
    for i, chunk in enumerate(retrieved_chunks, start=1):
        print(f"\nChunk {i}:")
        print(chunk)

    # Build context
    context = "\n\n".join(retrieved_chunks)

    # Build prompt
    prompt = f"""
Answer the question using only the context below.
If the answer is not in the context, say: "The information is not available in the provided context."

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer with GPT
    response = client.responses.create(
        model=chat_deployment,
        input=prompt,
        temperature=0
    )

    print("\nLLM Answer:")
    print(response.output_text)
    print()

Question: Is Great Wall of China visible from the Moon?
